In [ ]:
#========================
#AIPL 2026 Predection ML
#========================
#ALL librires are imported here

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import log_loss

from lightgbm import LGBMClassifier

import warnings
warnings.filterwarnings("ignore")

In [ ]:
#========================
#LOAD the data
#========================
train = pd.read_csv("train_IPL.csv")
schedule = pd.read_csv("schedule.csv")
sample = pd.read_csv("sample_submission.csv")

print(train.shape)
print(schedule.shape)
print(sample.shape)

train.head()


In [ ]:
#========================
#Create the Match data
#========================
# First innings final score
inn1 = (
    train[train["Innings"] == 1]
    .groupby("Match ID")
    .tail(1)[["Match ID", "Innings Runs"]]
    .rename(columns={"Innings Runs": "inn1_runs"})
)

# Second innings final score
inn2 = (
    train[train["Innings"] == 2]
    .groupby("Match ID")
    .tail(1)[["Match ID", "Innings Runs", "Innings Wickets"]]
    .rename(columns={
        "Innings Runs": "inn2_runs",
        "Innings Wickets": "inn2_wickets"
    })
)

# Match basic info
match_df = (
    train.groupby("Match ID")
    .first()
    .reset_index()
)
# Merge all data
match_df = match_df.merge(inn1, on="Match ID", how="left")
match_df = match_df.merge(inn2, on="Match ID", how="left")

print(match_df.shape)

match_df.head()


In [ ]:
#========================
# 4. REMOVE TIES / NO RESULTS
#========================

match_df = match_df[
    (~match_df["result_type"].isin(["tie", "no result"]))
]

print(match_df.shape)

In [ ]:
# ===========================
# 5. CREATE TARGET LABELS
# ===========================

def create_label(row):

    winner = row["match_won_by"]

    team_a = row["Bat First"]
    team_b = row["Bat Second"]

    inn1_runs = row["inn1_runs"]
    inn2_runs = row["inn2_runs"]

    wickets_lost = row["inn2_wickets"]

     # Team A wins
    if winner == team_a:

        margin = inn1_runs - inn2_runs

        if margin <= 20:
            return "A_small"
        else:
            return "A_big"
    # Team B wins
    elif winner == team_b:

        wickets_remaining = 10 - wickets_lost

        if wickets_remaining <= 5:
            return "B_small"
        else:
            return "B_big"

    else:
        return np.nan
    
    
match_df["target"] = match_df.apply(create_label, axis=1)

match_df = match_df.dropna(subset=["target"])

print(match_df["target"].value_counts())

In [ ]:
#========================
# 6. FEATURE ENGINEERING
#========================

# Match level features
features = [
    "Venue",
    "city",
    "Bat First",
    "Bat Second",
    "toss_winner",
    "toss_decision",
    "season"
]

data = match_df[features + ["target"]].copy()

data.head()

In [ ]:
#========================
# 7. LABEL ENCODING
#========================

encoders = {}

for col in features:

    le = LabelEncoder()

    data[col] = le.fit_transform(data[col].astype(str))

    encoders[col] = le


# Encode target
target_encoder = LabelEncoder()

data["target"] = target_encoder.fit_transform(data["target"])

print(target_encoder.classes_)

In [ ]:
#========================
# 8. TRAIN TEST SPLIT
#========================

X = data[features]
y = data["target"]

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_valid.shape)



In [ ]:
#========================
# 9. TRAIN MODEL
#========================
model = LGBMClassifier(
    objective="multiclass",
    num_class=4,
    n_estimators=300,
    learning_rate=0.03,
    max_depth=6,
    random_state=42
)

model.fit(X_train, y_train)

In [ ]:
#========================
# 10. VALIDATION
#========================
preds = model.predict_proba(X_valid)

score = log_loss(y_valid, preds)

print("Validation Log Loss:", score)

In [ ]:
#========================
# 11. PREPARE FUTURE MATCH DATA
#========================

schedule.head()
# Example expected columns:
#

future = schedule.copy()

# Dummy toss placeholders
future["toss_winner"] = future["team_a"]
future["toss_decision"] = "field"

future = future.rename(columns={
    "team_a": "Bat First",
    "team_b": "Bat Second"
})
future["toss_decision"] = "field"

future = future.rename(columns={
    "Team_A": "Bat First",
    "Team_B": "Bat Second"
})

In [ ]:
#========================
# 12. APPLY ENCODERS
#========================

future["Venue"] = future["venue"]
future["season"] = future["date"].astype(str).str[:4]

future_data = future[features].copy()

for col in features:

    le = encoders[col]

    # Handle unseen labels
    future_data[col] = future_data[col].astype(str)

    known = set(le.classes_)

    future_data[col] = future_data[col].apply(
        lambda x: x if x in known else le.classes_[0]
    )

    future_data[col] = le.transform(future_data[col])


future_data.head()

In [ ]:
#========================
# 13. PREDICT PROBABILITIES
#========================
future_preds = model.predict_proba(future_data)

future_preds[:5]

In [ ]:
# ============================================================
# 14. CREATE SUBMISSION
# ============================================================

submission = pd.DataFrame()

submission["match_id"] = schedule["match_id"]

class_names = target_encoder.classes_

for i, cls in enumerate(class_names):
    submission[cls] = future_preds[:, i]

submission.head()


In [ ]:
# ============================================================
# 15. ENSURE ALL 4 COLUMNS EXIST
# ============================================================

required_cols = ["A_small", "A_big", "B_small", "B_big"]

for col in required_cols:

    if col not in submission.columns:
        submission[col] = 0.0


submission = submission[
    ["match_id", "A_small", "A_big", "B_small", "B_big"]
]

submission.head()


In [ ]:
# ============================================================
# CREATE FINAL SUBMISSION
# ============================================================

submission = sample.copy()

# Predict probabilities for future matches
future_preds = model.predict_proba(future_data)

# Fill ONLY the last 5 rows
# (Future IPL 2026 fixtures)

submission.loc[
    submission.index[-5:],
    ["A_small", "A_big", "B_small", "B_big"]
] = future_preds

# ------------------------------------------------------------
# For remaining 48 rows
# Use class prior probabilities
# ------------------------------------------------------------

class_probs = (
    match_df["target"]
    .value_counts(normalize=True)
)

submission.loc[
    submission.index[:-5],
    "A_small"
] = class_probs.get("A_small", 0)

submission.loc[
    submission.index[:-5],
    "A_big"
] = class_probs.get("A_big", 0)

submission.loc[
    submission.index[:-5],
    "B_small"
] = class_probs.get("B_small", 0)

submission.loc[
    submission.index[:-5],
    "B_big"
] = class_probs.get("B_big", 0)

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

with open("submission.csv", "w", newline="", encoding="utf-8") as f:
    submission.to_csv(f, index=False)

print(submission.shape)
print("submission.csv created successfully")

In [ ]:
# ============================================================
# 17. FEATURE IMPORTANCE
# ============================================================

importance = pd.DataFrame({
    "Feature": features,
    "Importance": model.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print(importance)